# The Blub Paradox

## A junior with a 50-million-row problem
A team is uploading fuel transactions. Lots of them. Some are valid, some
are duplicates of previous failed attempts, some are quarantined for
manual review, some are rejected outright. The pipeline is small, the
business rules are real, and the cost of a bug is measured in misposted
fuel charges.

A junior on the team writes the first version. It builds. The tests pass.
The happy path works. Three weeks later, a batch run in production
**crashes the whole upload** on the first row whose vehicle isn't in the
lookup table. Another month later, a quieter bug is found: the aggressive
recovery mode silently skips rows it was supposed to reprocess. Three
weeks after *that*, a consumer of the response API breaks because someone
typo'd the status string.

None of these are bugs of *logic*. The logic is fine. They are bugs of
**what the language let the junior express**.

## What this book is

This book sets up a deliberately junior-to-mid level C# implementation of
that fuel-upload engine, walks through the seven bugs it lets you ship,
and then climbs the language ladder one rung at a time — idiomatic C#,
F#, Haskell, Rust — showing **which class of bug each rung makes
impossible**.

## Paul Graham's blub paradox

In a 2001 essay, Paul Graham described a paradox programmers fall into:

> Programmers, when they compare languages, judge them by their own
> language as the standard. If they don't know language X, then X looks
> hard to learn, redundant in features they already have, or even —
> paradoxically — *missing* features they don't realise they're missing.

He called the hypothetical mid-power language "Blub." A Blub programmer
looks **down** the power scale and sees obvious deficiencies in weaker
languages. A Blub programmer looking **up** sees only weirdness — pointless
abstraction, syntactic noise, things they could "do anyway" with enough
care.

The paradox is that **you cannot see what your language is missing while
you're inside it**. Higher-power languages don't make you write less code;
they make whole categories of mistake **structurally impossible**. From
inside the lower-power language you can't see the categories, because
your language doesn't *talk about* them.

## The fuel engine, five times

We picked a small but real domain: parse fuel rows → validate them →
check the vehicle exists → check for duplicates → apply a recovery policy
→ either accept, accept-with-warnings, quarantine, skip, or reject the
row → summarise the batch. The pipeline shape is the same in all five
implementations:

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 1330.9375 590" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381574567_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381574567_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381574567_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381574567_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381574567_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381574567_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M109.797,295L113.964,295C118.13,295,126.464,295,134.13,295C141.797,295,148.797,295,152.297,295L155.797,295" id="L_Parse_Validate_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Parse_Validate_0" data-points="W3sieCI6MTA5Ljc5Njg3NSwieSI6Mjk1fSx7IngiOjEzNC43OTY4NzUsInkiOjI5NX0seyJ4IjoxNTkuNzk2ODc1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M276.438,295L280.604,295C284.771,295,293.104,295,300.771,295C308.438,295,315.438,295,318.938,295L322.438,295" id="L_Validate_Vehicle_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Validate_Vehicle_0" data-points="W3sieCI6Mjc2LjQzNzUsInkiOjI5NX0seyJ4IjozMDEuNDM3NSwieSI6Mjk1fSx7IngiOjMyNi40Mzc1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M489.625,295L493.792,295C497.958,295,506.292,295,513.958,295C521.625,295,528.625,295,532.125,295L535.625,295" id="L_Vehicle_Duplicate_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Vehicle_Duplicate_0" data-points="W3sieCI6NDg5LjYyNSwieSI6Mjk1fSx7IngiOjUxNC42MjUsInkiOjI5NX0seyJ4Ijo1MzkuNjI1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M712.578,295L716.745,295C720.911,295,729.245,295,736.911,295C744.578,295,751.578,295,755.078,295L758.578,295" id="L_Duplicate_Decide_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Duplicate_Decide_0" data-points="W3sieCI6NzEyLjU3ODEyNSwieSI6Mjk1fSx7IngiOjczNy41NzgxMjUsInkiOjI5NX0seyJ4Ijo3NjIuNTc4MTI1LCJ5IjoyOTV9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M826.333,254.942L837.176,218.285C848.019,181.628,869.705,108.314,892.496,71.657C915.286,35,939.182,35,951.13,35L963.078,35" id="L_Decide_Accept_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_Accept_0" data-points="W3sieCI6ODI2LjMzMzExNjQyMDA5MDgsInkiOjI1NC45NDI0OTE0MjAwOTA5fSx7IngiOjg5MS4zOTA2MjUsInkiOjM1fSx7IngiOjk2Ny4wNzgxMjUsInkiOjM1fV0=" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M831.624,260.233L841.585,240.028C851.546,219.822,871.468,179.411,884.93,159.206C898.391,139,905.391,139,908.891,139L912.391,139" id="L_Decide_AcceptWarn_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_AcceptWarn_0" data-points="W3sieCI6ODMxLjYyMzk1Mzg2MDg2MTQsInkiOjI2MC4yMzMzMjg4NjA4NjE0fSx7IngiOjg5MS4zOTA2MjUsInkiOjEzOX0seyJ4Ijo5MTYuMzkwNjI1LCJ5IjoxMzl9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M845.452,274.061L853.108,268.884C860.765,263.708,876.078,253.354,893.903,248.177C911.729,243,932.068,243,942.237,243L952.406,243" id="L_Decide_Quarantine_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_Quarantine_0" data-points="W3sieCI6ODQ1LjQ1MTk1ODMzMzMzMzMsInkiOjI3NC4wNjEzMzMzMzMzMzMzfSx7IngiOjg5MS4zOTA2MjUsInkiOjI0M30seyJ4Ijo5NTYuNDA2MjUsInkiOjI0M31d" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M845.452,315.939L853.108,321.116C860.765,326.292,876.078,336.646,890.716,341.823C905.354,347,919.318,347,926.299,347L933.281,347" id="L_Decide_Skip_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_Skip_0" data-points="W3sieCI6ODQ1LjQ1MTk1ODMzMzMzMzMsInkiOjMxNS45Mzg2NjY2NjY2NjY3fSx7IngiOjg5MS4zOTA2MjUsInkiOjM0N30seyJ4Ijo5MzcuMjgxMjUsInkiOjM0N31d" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M831.624,329.767L841.585,349.972C851.546,370.178,871.468,410.589,893.674,430.794C915.88,451,940.37,451,952.615,451L964.859,451" id="L_Decide_Reject_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_Reject_0" data-points="W3sieCI6ODMxLjYyMzk1Mzg2MDg2MTQsInkiOjMyOS43NjY2NzExMzkxMzg2fSx7IngiOjg5MS4zOTA2MjUsInkiOjQ1MX0seyJ4Ijo5NjguODU5Mzc1LCJ5Ijo0NTF9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M826.333,335.058L837.176,371.715C848.019,408.372,869.705,481.686,895.091,518.343C920.477,555,949.563,555,964.105,555L978.648,555" id="L_Decide_Fatal_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Decide_Fatal_0" data-points="W3sieCI6ODI2LjMzMzExNjQyMDA5MDgsInkiOjMzNS4wNTc1MDg1Nzk5MDkxfSx7IngiOjg5MS4zOTA2MjUsInkiOjU1NX0seyJ4Ijo5ODIuNjQ4NDM3NSwieSI6NTU1fV0=" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1093.797,35L1106.411,35C1119.026,35,1144.255,35,1169.98,73.203C1195.705,111.406,1221.926,187.811,1235.036,226.014L1248.147,264.217" id="L_Accept_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Accept_Summary_0" data-points="W3sieCI6MTA5My43OTY4NzUsInkiOjM1fSx7IngiOjExNjkuNDg0Mzc1LCJ5IjozNX0seyJ4IjoxMjQ5LjQ0NTEwMjE2MzQ2MTYsInkiOjI2OH1d" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1144.484,139L1148.651,139C1152.818,139,1161.151,139,1177.284,159.921C1193.417,180.843,1217.349,222.685,1229.316,243.607L1241.282,264.528" id="L_AcceptWarn_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_AcceptWarn_Summary_0" data-points="W3sieCI6MTE0NC40ODQzNzUsInkiOjEzOX0seyJ4IjoxMTY5LjQ4NDM3NSwieSI6MTM5fSx7IngiOjEyNDMuMjY3ODc4NjA1NzY5MywieSI6MjY4fV0=" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1104.469,243L1115.305,243C1126.141,243,1147.813,243,1165.222,246.831C1182.632,250.662,1195.779,258.324,1202.352,262.155L1208.926,265.986" id="L_Quarantine_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Quarantine_Summary_0" data-points="W3sieCI6MTEwNC40Njg3NSwieSI6MjQzfSx7IngiOjExNjkuNDg0Mzc1LCJ5IjoyNDN9LHsieCI6MTIxMi4zODE3NjA4MTczMDc2LCJ5IjoyNjh9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1123.594,347L1131.242,347C1138.891,347,1154.188,347,1168.41,343.169C1182.632,339.338,1195.779,331.676,1202.352,327.845L1208.926,324.014" id="L_Skip_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Skip_Summary_0" data-points="W3sieCI6MTEyMy41OTM3NSwieSI6MzQ3fSx7IngiOjExNjkuNDg0Mzc1LCJ5IjozNDd9LHsieCI6MTIxMi4zODE3NjA4MTczMDc2LCJ5IjozMjJ9XQ==" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1092.016,451L1104.927,451C1117.839,451,1143.661,451,1168.539,430.079C1193.417,409.157,1217.349,367.315,1229.316,346.393L1241.282,325.472" id="L_Reject_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Reject_Summary_0" data-points="W3sieCI6MTA5Mi4wMTU2MjUsInkiOjQ1MX0seyJ4IjoxMTY5LjQ4NDM3NSwieSI6NDUxfSx7IngiOjEyNDMuMjY3ODc4NjA1NzY5MywieSI6MzIyfV0=" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path><path d="M1078.227,555L1093.436,555C1108.646,555,1139.065,555,1167.385,516.797C1195.705,478.594,1221.926,402.189,1235.036,363.986L1248.147,325.783" id="L_Fatal_Summary_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_Fatal_Summary_0" data-points="W3sieCI6MTA3OC4yMjY1NjI1LCJ5Ijo1NTV9LHsieCI6MTE2OS40ODQzNzUsInkiOjU1NX0seyJ4IjoxMjQ5LjQ0NTEwMjE2MzQ2MTYsInkiOjMyMn1d" marker-end="url(#mermaid-1779381574567_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_Parse_Validate_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Validate_Vehicle_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Vehicle_Duplicate_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Duplicate_Decide_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_Accept_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_AcceptWarn_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_Quarantine_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_Skip_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_Reject_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Decide_Fatal_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Accept_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_AcceptWarn_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Quarantine_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Skip_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Reject_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_Fatal_Summary_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-Parse-0" transform="translate(58.8984375, 295)"><rect class="basic label-container" style="" x="-50.8984375" y="-27" width="101.796875" height="54"></rect><g class="label" style="" transform="translate(-20.8984375, -12)"><rect></rect><foreignobject width="41.796875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Parse
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Validate-1" transform="translate(218.1171875, 295)"><rect class="basic label-container" style="" x="-58.3203125" y="-27" width="116.640625" height="54"></rect><g class="label" style="" transform="translate(-28.3203125, -12)"><rect></rect><foreignobject width="56.640625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Validate
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Vehicle-2" transform="translate(408.03125, 295)"><rect class="basic label-container" style="" x="-81.59375" y="-27" width="163.1875" height="54"></rect><g class="label" style="" transform="translate(-51.59375, -12)"><rect></rect><foreignobject width="103.1875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Vehicle lookup
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Duplicate-3" transform="translate(626.1015625, 295)"><rect class="basic label-container" style="" x="-86.4765625" y="-27" width="172.953125" height="54"></rect><g class="label" style="" transform="translate(-56.4765625, -12)"><rect></rect><foreignobject width="112.953125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Duplicate check
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Decide-5" transform="translate(814.484375, 295)"><polygon points="51.90625,0 103.8125,-51.90625 51.90625,-103.8125 0,-51.90625" class="label-container" transform="translate(-51.40625, 51.90625)"></polygon><g class="label" style="" transform="translate(-24.90625, -12)"><rect></rect><foreignobject width="49.8125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Decide
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Accept-7" transform="translate(1030.4375, 35)"><rect class="basic label-container" style="" x="-63.359375" y="-27" width="126.71875" height="54"></rect><g class="label" style="" transform="translate(-33.359375, -12)"><rect></rect><foreignobject width="66.71875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Accepted
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-AcceptWarn-9" transform="translate(1030.4375, 139)"><rect class="basic label-container" style="" x="-114.046875" y="-27" width="228.09375" height="54"></rect><g class="label" style="" transform="translate(-84.046875, -12)"><rect></rect><foreignobject width="168.09375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Accepted with warnings
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Quarantine-11" transform="translate(1030.4375, 243)"><rect class="basic label-container" style="" x="-74.03125" y="-27" width="148.0625" height="54"></rect><g class="label" style="" transform="translate(-44.03125, -12)"><rect></rect><foreignobject width="88.0625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Quarantined
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Skip-13" transform="translate(1030.4375, 347)"><rect class="basic label-container" style="" x="-93.15625" y="-27" width="186.3125" height="54"></rect><g class="label" style="" transform="translate(-63.15625, -12)"><rect></rect><foreignobject width="126.3125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Skipped duplicate
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Reject-15" transform="translate(1030.4375, 451)"><rect class="basic label-container" style="" x="-61.578125" y="-27" width="123.15625" height="54"></rect><g class="label" style="" transform="translate(-31.578125, -12)"><rect></rect><foreignobject width="63.15625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Rejected
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Fatal-17" transform="translate(1030.4375, 555)"><rect class="basic label-container" style="" x="-47.7890625" y="-27" width="95.578125" height="54"></rect><g class="label" style="" transform="translate(-17.7890625, -12)"><rect></rect><foreignobject width="35.578125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Fatal
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-Summary-19" transform="translate(1258.7109375, 295)"><rect class="basic label-container" style="" x="-64.2265625" y="-27" width="128.453125" height="54"></rect><g class="label" style="" transform="translate(-34.2265625, -12)"><rect></rect><foreignobject width="68.453125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Summary
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

What differs across the five implementations is **the type system you
have to express that pipeline in**.

## The ladder

We're going to climb it in this order:

1. **Normal junior C#** — the version that ships first. No nullable
   reference types, mutable POCOs everywhere, string-typed status, exception
   for control flow. We document the [seven footguns](03-seven-footguns.ipynb)
   it makes easy.

2. **Idiomatic C#** — same language, modern features turned on.
   Sealed-record discriminated unions, NRT, switch expressions on type
   patterns, `IReadOnlyList<T>`. Closes six of the seven footguns.
   [Read it →](04-idiomatic-csharp.ipynb)

3. **F#** — leaves C# behind. Native discriminated unions, exhaustive
   pattern matching that the compiler *enforces*, immutability by default,
   no nulls in F#'s own types. The footgun list shrinks again.
   [Read it →](05-fsharp-native.ipynb)

4. **Haskell** — the type-driven extreme. Algebraic data types are the
   *natural* way to model anything. `NonEmpty` lets you say "this list has
   at least one element" *at the type level*. Side effects are tracked in
   the signature.
   [Read it →](07-haskell-primer.ipynb)

5. **Rust** — same algebraic types as Haskell, plus *ownership*. Memory
   safety without a garbage collector. Exhaustive `match` enforced by the
   compiler. A different goal (systems programming) but lands in nearly the
   same place on this domain.
   [Read it →](09-rust-lands.ipynb)

At each rung we ask the same four questions:

- What does this rung **protect against** that the one below didn't?
- What can **still** go wrong?
- Is it **easier or harder** to read?
- How easy is it to **extend** — to add a 7th `RowDecision` case?

## What the book is not

It is not a polemic for any particular language. The idiomatic C# branch
closes six of seven footguns; F# closes them more cheaply; Haskell makes
the seventh structurally impossible at the cost of a steeper learning
curve; Rust pays for the same guarantees with the borrow checker. There
is no free lunch on this ladder. The point is to **see** the tradeoffs,
not to ratify a winner.

## How to use it

If you have an hour, read Part I straight through. The blub-paradox
journey lands in one sitting.

If you want a reference, jump to [Part II](../part2/r1-decision.ipynb) — each
chapter is one feature (modelling decisions, validation, recovery,
exhaustiveness, mutability, null, the boundary) shown five times side-by-side.

If you're skeptical of any chapter's claim, the source code is in the same
repository under `csharp-fuel-engine/`, `fsharp-fuel-engine/`,
`haskell-fuel-engine/`, and `rust-fuel-engine/`. Build instructions are in
[Setup](../setup.ipynb).

Let's start with the version that ships first.